#
Multimodal Emotion Recognition
### TESS Dataset — Speech + Text + Fusion Pipelines
**CNN + BiLSTM + Attention | DistilBERT | Concatenation Fusion**


## ⚙️ 0. Environment Setup

In [1]:
# Install all dependencies
!pip install -q tensorflow==2.15.0 transformers==4.38.2 librosa==0.10.1 \
    soundfile==0.12.1 kaggle seaborn scikit-learn torch tqdm

import os

# ── Kaggle API key ────────────────────────────────────────────────────────
# Option A: upload kaggle.json via Colab file picker
# from google.colab import files; files.upload()

# Option B: paste credentials directly (recommended for automation)
os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
KAGGLE_USERNAME = "YOUR_KAGGLE_USERNAME"   # <-- change this
KAGGLE_KEY      = "YOUR_KAGGLE_API_KEY"    # <-- change this
with open(os.path.expanduser("~/.kaggle/kaggle.json"), "w") as f:
    import json
    json.dump({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}, f)
os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
print("Kaggle credentials set.")

ERROR: Could not find a version that satisfies the requirement tensorflow==2.15.0 (from versions: 2.16.0rc0, 2.16.1, 2.16.2, 2.17.0rc0, 2.17.0rc1, 2.17.0, 2.17.1, 2.18.0rc0, 2.18.0rc1, 2.18.0rc2, 2.18.0, 2.18.1, 2.19.0rc0, 2.19.0, 2.19.1, 2.20.0rc0, 2.20.0, 2.21.0rc0, 2.21.0rc1, 2.21.0)
ERROR: No matching distribution found for tensorflow==2.15.0
Kaggle credentials set.


In [2]:
# Download TESS dataset from Kaggle
!kaggle datasets download -d ejlok1/toronto-emotional-speech-set-tess \
    -p /content/tess --unzip

import os
os.environ["TESS_DIR"] = "/content/tess"

# Verify download
wav_count = sum(1 for r, d, fs in os.walk("/content/tess") for f in fs if f.endswith(".wav"))
print(f"WAV files found: {wav_count}")
assert wav_count > 0, "Dataset not found — check Kaggle credentials"

# Create all output directories
for d in [
    "/content/project/Results/accuracy_tables",
    "/content/project/Results/plots",
    "/content/project/models/speech_pipeline",
    "/content/project/models/text_pipeline/text_model",
    "/content/project/models/fusion_pipeline",
]:
    os.makedirs(d, exist_ok=True)
print("Directories ready.")

Dataset URL: https://www.kaggle.com/datasets/ejlok1/toronto-emotional-speech-set-tess
License(s): Attribution-NonCommercial-NoDerivatives 4.0 International (CC BY-NC-ND 4.0)
100% 428M/428M [00:03<00:00, 128MB/s]

WAV files found: 5600
Directories ready.


### 🔒 Data-Leak Sanity Check

In [3]:
# ═══════════════════════════════════════════════════════════
#  DATA-LEAK SANITY CHECK — run after each training cell
#  Must print "Overlap: 0" for all three pipelines
# ═══════════════════════════════════════════════════════════
import os, glob, numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

SEED     = 42
TESS_DIR = os.environ.get("TESS_DIR", "/content/project/tess")
EMOTIONS = ["angry","disgust","fear","happy","neutral","ps","sad"]

records = []
wav_files = glob.glob(os.path.join(TESS_DIR, "**", "*.wav"), recursive=True)
for fp in wav_files:
    folder  = os.path.basename(os.path.dirname(fp)).lower()
    emotion = folder.split("_")[-1]
    if emotion in EMOTIONS:
        records.append({"path": fp, "emotion": emotion})

df = pd.DataFrame(records)
le = LabelEncoder(); df["label"] = le.fit_transform(df["emotion"])

train_df, test_df = train_test_split(df, test_size=0.15, random_state=SEED,
                                      stratify=df["label"])
train_df, val_df  = train_test_split(train_df, test_size=0.15, random_state=SEED,
                                      stratify=train_df["label"])

train_paths = set(train_df["path"])
val_paths   = set(val_df["path"])
test_paths  = set(test_df["path"])

tv_overlap  = train_paths & val_paths
tt_overlap  = train_paths & test_paths
vt_overlap  = val_paths   & test_paths

print(f"Train/Val  overlap : {len(tv_overlap)} (must be 0)")
print(f"Train/Test overlap : {len(tt_overlap)} (must be 0)")
print(f"Val/Test   overlap : {len(vt_overlap)} (must be 0)")
print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
assert len(tv_overlap) == 0, "DATA LEAK: train/val overlap!"
assert len(tt_overlap) == 0, "DATA LEAK: train/test overlap!"
assert len(vt_overlap) == 0, "DATA LEAK: val/test overlap!"
print("✓ All splits are clean — no data leakage.")


Train/Val  overlap : 0 (must be 0)
Train/Test overlap : 0 (must be 0)
Val/Test   overlap : 0 (must be 0)
Train: 3468 | Val: 612 | Test: 720
✓ All splits are clean — no data leakage.


---
## 🎙️ Part 1 — Speech Emotion Pipeline

| Block | Choice |
|-------|--------|
| Preprocessing | Silence trim, pad/crop to 3s, SR=22050 |
| Feature Extraction | 40 MFCC + 40 Δ-MFCC + 128 mel + 12 chroma + 7 spectral contrast + 1 ZCR |
| Augmentation | Pitch shift ±2, noise σ=0.005, time stretch 0.85–1.15, amplitude scaling |
| Temporal Modelling | CNN (32→64→128 filters) + BiLSTM (128+64) + Soft Attention |
| Classifier | Dense(128) → Dense(64) → Softmax(7) |

### 1a. Speech — Training

In [4]:
"""
Speech Emotion Recognition — CNN + BiLSTM + Attention
Dataset: TESS (Toronto Emotional Speech Set)
Self-contained: preprocessing, augmentation, feature extraction,
model creation, training, evaluation, plotting, saving.
"""

# ── Colab: install deps if needed ──────────────────────────────────────────
import subprocess, sys
def _install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

try:
    import librosa
except ImportError:
    _install("librosa")
try:
    import soundfile
except ImportError:
    _install("soundfile")

# ── Imports ────────────────────────────────────────────────────────────────
import os, glob, random, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

import librosa
import librosa.display
import soundfile as sf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, roc_curve, auc)
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, backend as K
from tensorflow.keras.utils import to_categorical

# ── Reproducibility ────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

# ── GPU config ─────────────────────────────────────────────────────────────
gpus = tf.config.list_physical_devices("GPU")
for g in gpus:
    tf.config.experimental.set_memory_growth(g, True)
print(f"GPUs available: {len(gpus)}")

# ── Paths ──────────────────────────────────────────────────────────────────
BASE_DIR   = "/content/project"
TESS_DIR   = os.environ.get("TESS_DIR", os.path.join(BASE_DIR, "tess"))
MODEL_DIR  = "/content/project/models/speech_pipeline"
RESULTS_AT = os.path.join(BASE_DIR, "Results", "accuracy_tables")
RESULTS_PL = os.path.join(BASE_DIR, "Results", "plots")
for d in [RESULTS_AT, RESULTS_PL, MODEL_DIR]:
    os.makedirs(d, exist_ok=True)

# ── Hyperparameters ────────────────────────────────────────────────────────
SR         = 22050
DURATION   = 3.0        # seconds per clip
N_MFCC     = 40
N_MELS     = 128
HOP_LENGTH = 512
N_FFT      = 2048
MAX_LEN    = 128        # time frames
BATCH_SIZE = 32
EPOCHS     = 60
LR         = 1e-3
EMOTIONS   = ["angry","disgust","fear","happy","neutral","ps","sad"]

# ── Feature extraction ─────────────────────────────────────────────────────
def extract_features(audio, sr):
    """
    Returns array shape (MAX_LEN, 180):
      40 MFCC + 40 delta MFCC + 128 mel + 12 chroma + 7 spectral_contrast + 1 ZCR + ??? → 228
    We keep first 180 dims for fixed size.
    """
    # ── pad / trim ─────────────────────────────────────────────────────────
    target = int(DURATION * sr)
    if len(audio) < target:
        audio = np.pad(audio, (0, target - len(audio)))
    else:
        audio = audio[:target]

    # ── MFCC (40) ──────────────────────────────────────────────────────────
    mfcc  = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=N_MFCC,
                                  n_fft=N_FFT, hop_length=HOP_LENGTH)          # (40, T)
    delta = librosa.feature.delta(mfcc)                                         # (40, T)

    # ── Mel spectrogram (128) ───────────────────────────────────────────────
    mel   = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=N_MELS,
                                            n_fft=N_FFT, hop_length=HOP_LENGTH) # (128, T)
    mel   = librosa.power_to_db(mel, ref=np.max)

    # ── Chroma (12) ────────────────────────────────────────────────────────
    chroma = librosa.feature.chroma_stft(y=audio, sr=sr, n_fft=N_FFT,
                                          hop_length=HOP_LENGTH)                 # (12, T)

    # ── Spectral contrast (7) ──────────────────────────────────────────────
    sc     = librosa.feature.spectral_contrast(y=audio, sr=sr, n_fft=N_FFT,
                                                hop_length=HOP_LENGTH)           # (7, T)

    # ── ZCR (1) ────────────────────────────────────────────────────────────
    zcr    = librosa.feature.zero_crossing_rate(audio, hop_length=HOP_LENGTH)   # (1, T)

    # ── Stack → (228, T) → transpose → (T, 228) ────────────────────────────
    feat = np.vstack([mfcc, delta, mel, chroma, sc, zcr])  # (228, T)
    feat = feat.T                                           # (T, 228)

    # ── Pad/trim time axis ─────────────────────────────────────────────────
    if feat.shape[0] < MAX_LEN:
        feat = np.pad(feat, ((0, MAX_LEN - feat.shape[0]), (0, 0)))
    else:
        feat = feat[:MAX_LEN, :]

    # ── Normalise per feature ──────────────────────────────────────────────
    mean = feat.mean(axis=0, keepdims=True)
    std  = feat.std(axis=0, keepdims=True) + 1e-8
    feat = (feat - mean) / std

    return feat[:, :180]   # fixed 180 features

# ── Augmentation ───────────────────────────────────────────────────────────
def augment(audio, sr):
    """Apply one of 4 augmentations randomly."""
    choice = random.randint(0, 3)
    if choice == 0:                       # noise injection
        noise = np.random.randn(len(audio)) * 0.005
        audio = audio + noise
    elif choice == 1:                     # pitch shift
        n_steps = random.uniform(-2, 2)
        audio   = librosa.effects.pitch_shift(audio, sr=sr, n_steps=n_steps)
    elif choice == 2:                     # time stretch
        rate  = random.uniform(0.85, 1.15)
        audio = librosa.effects.time_stretch(audio, rate=rate)
    elif choice == 3:                     # amplitude scaling
        audio = audio * random.uniform(0.7, 1.3)
    return audio

# ── Dataset loading ────────────────────────────────────────────────────────
def load_tess(tess_dir):
    """Walk TESS folder structure, load audio + labels."""
    records = []
    # TESS layout: <tess_dir>/<FolderName>/<actor>_<word>_<emotion>.wav
    # Folder names contain emotion in their name, e.g. "OAF_angry"
    wav_files = glob.glob(os.path.join(tess_dir, "**", "*.wav"), recursive=True)
    if len(wav_files) == 0:
        raise FileNotFoundError(
            f"No .wav files found under {tess_dir}. "
            "Set TESS_DIR env variable or place dataset at project/tess/"
        )
    for fp in wav_files:
        folder = os.path.basename(os.path.dirname(fp)).lower()
        # emotion is last segment after last underscore in folder name
        emotion = folder.split("_")[-1]
        if emotion not in EMOTIONS:
            continue
        records.append({"path": fp, "emotion": emotion})
    df = pd.DataFrame(records)
    print(f"Loaded {len(df)} files | emotions: {df['emotion'].value_counts().to_dict()}")
    return df

# ── Build feature matrix ───────────────────────────────────────────────────
def build_dataset(df, augment_train=True, aug_ratio=0.5):
    X, y = [], []
    for _, row in df.iterrows():
        try:
            audio, sr = librosa.load(row["path"], sr=SR, mono=True)
            # trim leading/trailing silence
            audio, _ = librosa.effects.trim(audio, top_db=20)
            feat = extract_features(audio, sr)
            X.append(feat)
            y.append(row["emotion"])
            # augmentation for training data
            if augment_train and random.random() < aug_ratio:
                aug_audio = augment(audio, sr)
                aug_feat  = extract_features(aug_audio, sr)
                X.append(aug_feat)
                y.append(row["emotion"])
        except Exception as e:
            print(f"Skipping {row['path']}: {e}")
    return np.array(X, dtype=np.float32), np.array(y)

# ── Attention layer ────────────────────────────────────────────────────────
class AttentionLayer(layers.Layer):
    """Soft attention over time axis."""
    def __init__(self, units=64, **kwargs):
        super().__init__(**kwargs)
        self.W = layers.Dense(units, activation="tanh")
        self.V = layers.Dense(1)

    def call(self, x):               # x: (batch, time, features)
        score  = self.V(self.W(x))   # (batch, time, 1)
        weight = tf.nn.softmax(score, axis=1)
        ctx    = tf.reduce_sum(weight * x, axis=1)  # (batch, features)
        return ctx

# ── Model definition ───────────────────────────────────────────────────────
def build_model(input_shape, num_classes):
    """CNN + BiLSTM + Attention classifier."""
    inp = layers.Input(shape=input_shape, name="mfcc_input")

    # ── CNN block ──────────────────────────────────────────────────────────
    x = layers.Reshape((*input_shape, 1))(inp)          # add channel dim
    x = layers.Conv2D(32, (3,3), padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2,2))(x)
    x = layers.Dropout(0.25)(x)

    x = layers.Conv2D(64, (3,3), padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2,2))(x)
    x = layers.Dropout(0.25)(x)

    x = layers.Conv2D(128, (3,3), padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2,1))(x)
    x = layers.Dropout(0.25)(x)

    # ── Merge spatial dims into features ───────────────────────────────────
    # shape after pool3: (batch, T//8, F//4, 128) → reshape to (batch, T', feat)
    t_dim  = x.shape[1]
    f_dim  = x.shape[2]
    ch_dim = x.shape[3]
    x = layers.Reshape((t_dim, f_dim * ch_dim))(x)

    # ── BiLSTM ─────────────────────────────────────────────────────────────
    x = layers.Bidirectional(layers.LSTM(128, return_sequences=True,
                                          dropout=0.3, recurrent_dropout=0.2))(x)
    x = layers.Bidirectional(layers.LSTM(64,  return_sequences=True,
                                          dropout=0.3, recurrent_dropout=0.2))(x)

    # ── Attention ──────────────────────────────────────────────────────────
    x = AttentionLayer(units=64)(x)

    # ── Classifier head ────────────────────────────────────────────────────
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(64, activation="relu")(x)
    out = layers.Dense(num_classes, activation="softmax", name="emotion_output")(x)

    model = models.Model(inp, out, name="CNN_BiLSTM_Attention")
    return model

# ── Plot helpers ───────────────────────────────────────────────────────────
def plot_history(history, prefix):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(history.history["accuracy"],     label="train acc")
    axes[0].plot(history.history["val_accuracy"], label="val acc")
    axes[0].set_title("Accuracy"); axes[0].legend(); axes[0].set_xlabel("Epoch")
    axes[1].plot(history.history["loss"],     label="train loss")
    axes[1].plot(history.history["val_loss"], label="val loss")
    axes[1].set_title("Loss"); axes[1].legend(); axes[1].set_xlabel("Epoch")
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_PL, f"{prefix}_training_curves.png"), dpi=150)
    plt.close()

def plot_confusion(cm, labels, prefix):
    fig, ax = plt.subplots(figsize=(9, 7))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=labels, yticklabels=labels, ax=ax)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title(f"{prefix} Confusion Matrix")
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_PL, f"{prefix}_confusion_matrix.png"), dpi=150)
    plt.close()

def plot_roc(y_true_bin, y_score, labels, prefix):
    fig, ax = plt.subplots(figsize=(10, 7))
    for i, lbl in enumerate(labels):
        fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_score[:, i])
        roc_auc     = auc(fpr, tpr)
        ax.plot(fpr, tpr, label=f"{lbl} (AUC={roc_auc:.2f})")
    ax.plot([0,1],[0,1],"k--")
    ax.set_xlabel("FPR"); ax.set_ylabel("TPR")
    ax.set_title(f"{prefix} ROC Curve"); ax.legend(loc="lower right")
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_PL, f"{prefix}_roc_curve.png"), dpi=150)
    plt.close()

def save_metrics_csv(report_dict, prefix):
    rows = []
    for label, vals in report_dict.items():
        if isinstance(vals, dict):
            rows.append({"class": label, **vals})
    df = pd.DataFrame(rows)
    path = os.path.join(RESULTS_AT, f"{prefix}_metrics.csv")
    df.to_csv(path, index=False)
    print(f"Metrics saved → {path}")

# ── Main ───────────────────────────────────────────────────────────────────
def main():
    print("=" * 60)
    print("SPEECH EMOTION RECOGNITION — Training")
    print("=" * 60)

    # ── Load dataset ───────────────────────────────────────────────────────
    df = load_tess(TESS_DIR)

    # ── Encode labels ──────────────────────────────────────────────────────
    le          = LabelEncoder()
    df["label"] = le.fit_transform(df["emotion"])
    num_classes = len(le.classes_)
    print(f"Classes ({num_classes}): {list(le.classes_)}")

    # ── Train / val / test split ───────────────────────────────────────────
    train_df, test_df = train_test_split(df, test_size=0.15, random_state=SEED,
                                          stratify=df["label"])
    train_df, val_df  = train_test_split(train_df, test_size=0.15, random_state=SEED,
                                          stratify=train_df["label"])
    print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

    # ── Feature extraction ─────────────────────────────────────────────────
    print("Extracting features (train) …")
    X_train, y_train_str = build_dataset(train_df, augment_train=True)
    print("Extracting features (val) …")
    X_val,   y_val_str   = build_dataset(val_df,   augment_train=False)
    print("Extracting features (test) …")
    X_test,  y_test_str  = build_dataset(test_df,  augment_train=False)

    y_train = to_categorical(le.transform(y_train_str), num_classes)
    y_val   = to_categorical(le.transform(y_val_str),   num_classes)
    y_test  = to_categorical(le.transform(y_test_str),  num_classes)

    print(f"X_train: {X_train.shape} | X_val: {X_val.shape} | X_test: {X_test.shape}")

    # ── Class weights ──────────────────────────────────────────────────────
    cw = compute_class_weight("balanced", classes=np.arange(num_classes),
                               y=np.argmax(y_train, axis=1))
    class_weight = dict(enumerate(cw))

    # ── Build model ────────────────────────────────────────────────────────
    model = build_model(input_shape=X_train.shape[1:], num_classes=num_classes)
    model.summary()

    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LR),
                  loss="categorical_crossentropy",
                  metrics=["accuracy"])

    # ── Callbacks ──────────────────────────────────────────────────────────
    cb_list = [
        callbacks.EarlyStopping(monitor="val_accuracy", patience=12,
                                restore_best_weights=True, verbose=1),
        callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                    patience=6, min_lr=1e-6, verbose=1),
        callbacks.ModelCheckpoint(
            filepath=os.path.join(MODEL_DIR, "speech_model_best.h5"),
            monitor="val_accuracy", save_best_only=True, verbose=1),
    ]

    # ── Training ───────────────────────────────────────────────────────────
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        class_weight=class_weight,
        callbacks=cb_list,
        verbose=1,
    )

    # ── Save final model ───────────────────────────────────────────────────
    model.save(os.path.join(MODEL_DIR, "speech_model.h5"))
    np.save(os.path.join(MODEL_DIR, "label_classes.npy"), le.classes_)
    print("Model saved.")

    # ── Plots ──────────────────────────────────────────────────────────────
    plot_history(history, "speech")

    # ── Evaluation ────────────────────────────────────────────────────────
    y_pred_prob = model.predict(X_test, verbose=0)
    y_pred      = np.argmax(y_pred_prob, axis=1)
    y_true      = np.argmax(y_test, axis=1)

    acc = accuracy_score(y_true, y_pred)
    print(f"\nTest Accuracy: {acc:.4f}")

    report = classification_report(y_true, y_pred,
                                   target_names=le.classes_, output_dict=True)
    print(classification_report(y_true, y_pred, target_names=le.classes_))

    cm = confusion_matrix(y_true, y_pred)
    plot_confusion(cm, le.classes_, "speech")
    plot_roc(y_test, y_pred_prob, le.classes_, "speech")
    save_metrics_csv(report, "speech")

    # ── Save test predictions ──────────────────────────────────────────────
    pred_df = pd.DataFrame({
        "true_label": le.inverse_transform(y_true),
        "pred_label": le.inverse_transform(y_pred),
        "correct":    y_true == y_pred,
    })
    pred_df.to_csv(os.path.join(RESULTS_AT, "speech_predictions.csv"), index=False)
    print("Predictions saved.")

    print("\nSpeech pipeline training COMPLETE.")

if __name__ == "__main__":
    main()


main()

GPUs available: 1
SPEECH EMOTION RECOGNITION — Training
Loaded 4800 files | emotions: {'angry': 800, 'happy': 800, 'neutral': 800, 'fear': 800, 'sad': 800, 'disgust': 800}
Classes (6): ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad']
Train: 3468 | Val: 612 | Test: 720
Extracting features (train) …
Extracting features (val) …
Extracting features (test) …
X_train: (5188, 128, 180) | X_val: (612, 128, 180) | X_test: (720, 128, 180)


Model: "CNN_BiLSTM_Attention"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mfcc_input (InputLayer)         │ (None, 128, 180)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape (Reshape)               │ (None, 128, 180, 1)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 128, 180, 32)   │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128, 180, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 64, 90, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64, 90, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 64, 90, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64, 90, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 32, 45, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32, 45, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 32, 45, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 32, 45, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 16, 45, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 16, 45, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape_1 (Reshape)             │ (None, 16, 5760)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 16, 256)        │     6,030,336 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 16, 128)        │       164,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ attention_layer                 │ (None, 128)            │         8,321 │
│ (AttentionLayer)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ emotion_output (Dense)          │ (None, 6)              │           390 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,321,735 (24.12 MB)

 Trainable params: 6,321,287 (24.11 MB)

...

Epoch 1/60
164/164 ━━━━━━━━━━━━━━━━━━━━ 0s 278ms/step - accuracy: 0.5795 - loss: 1.0086
Epoch 1: val_accuracy improved from None to 0.27288, saving model to /content/project/models/speech_pipeline/speech_model_best.h5



Epoch 1: finished saving model to /content/project/models/speech_pipeline/speech_model_best.h5
164/164 ━━━━━━━━━━━━━━━━━━━━ 67s 301ms/step - accuracy: 0.7951 - loss: 0.5168 - val_accuracy: 0.2729 - val_loss: 2.8805 - learning_rate: 0.0010
Epoch 2/60
164/164 ━━━━━━━━━━━━━━━━━━━━ 0s 286ms/step - accuracy: 0.9894 - loss: 0.0379
Epoch 2: val_accuracy improved from 0.27288 to 0.68954, saving model to /content/project/models/speech_pipeline/speech_model_best.h5



Epoch 2: finished saving model to /content/project/models/speech_pipeline/speech_model_best.h5
164/164 ━━━━━━━━━━━━━━━━━━━━ 48s 293ms/step - accuracy: 0.9914 - loss: 0.0334 - val_accuracy: 0.6895 - val_loss: 1.5740 - learning_rate: 0.0010
Epoch 3/60
164/164 ━━━━━━━━━━━━━━━━━━━━ 0s 290ms/step - accuracy: 0.9896 - loss: 0.0320
Epoch 3: val_accuracy improved from 0.68954 to 1.00000, saving model to /content/project/models/speech_pipeline/speech_model_best.h5



Epoch 3: finished saving model to /content/project/models/speech_pipeline/speech_model_best.h5
164/164 ━━━━━━━━━━━━━━━━━━━━ 49s 299ms/step - accuracy: 0.9914 - loss: 0.0266 - val_accuracy: 1.0000 - val_loss: 0.0018 - learning_rate: 0.0010
Epoch 4/60
164/164 ━━━━━━━━━━━━━━━━━━━━ 0s 291ms/step - accuracy: 0.9913 - loss: 0.0304
Epoch 4: val_accuracy did not improve from 1.00000
164/164 ━━━━━━━━━━━━━━━━━━━━ 49s 299ms/step - accuracy: 0.9923 - loss: 0.0284 - val_accuracy: 1.0000 - val_loss: 2.8101e-04 - learning_rate: 0.0010
Epoch 5/60
164/164 ━━━━━━━━━━━━━━━━━━━━ 0s 292ms/step - accuracy: 0.9967 - loss: 0.0115
Epoch 5: val_accuracy did not improve from 1.00000
164/164 ━━━━━━━━━━━━━━━━━━━━ 49s 298ms/step - accuracy: 0.9965 - loss: 0.0133 - val_accuracy: 0.9984 - val_loss: 0.0089 - learning_rate: 0.0010
Epoch 6/60
164/164 ━━━━━━━━━━━━━━━━━━━━ 0s 292ms/step - accuracy: 0.9961 - loss: 0.0146
Epoch 6: val_accuracy did not improve from 1.00000
164/164 ━━━━━━━━━━━━━━━━━━━━ 82s 298ms/step - accur

Model saved.

Test Accuracy: 0.9958
              precision    recall  f1-score   support

       angry       1.00      0.97      0.99       120
     disgust       1.00      1.00      1.00       120
        fear       1.00      1.00      1.00       120
       happy       0.98      1.00      0.99       120
     neutral       1.00      1.00      1.00       120
         sad       1.00      1.00      1.00       120

    accuracy                           1.00       720
   macro avg       1.00      1.00      1.00       720
weighted avg       1.00      1.00      1.00       720

Metrics saved → /content/project/Results/accuracy_tables/speech_metrics.csv
Predictions saved.

Speech pipeline training COMPLETE.


### 1b. Speech — Testing & Evaluation

In [5]:
import os, glob, random, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import librosa

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (classification_report, confusion_matrix,
                              accuracy_score, roc_curve, auc)
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from tensorflow.keras import backend as K
from tensorflow.keras import layers, models

# --- Attention layer (copied from train.py) --------------------------------
class AttentionLayer(layers.Layer):
    """Soft attention over time axis."""
    def __init__(self, units=64, **kwargs):
        super().__init__(**kwargs)
        self.W = layers.Dense(units, activation="tanh")
        self.V = layers.Dense(1)

    def call(self, x):               # x: (batch, time, features)
        score  = self.V(self.W(x))   # (batch, time, 1)
        weight = tf.nn.softmax(score, axis=1)
        ctx    = tf.reduce_sum(weight * x, axis=1)  # (batch, features)
        return ctx

# ── Reproducibility ────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

# ── Paths ──────────────────────────────────────────────────────────────────
BASE_DIR   = "/content/project"
TESS_DIR   = os.environ.get("TESS_DIR", os.path.join(BASE_DIR, "tess"))
MODEL_DIR  = "/content/project/models/speech_pipeline"
RESULTS_AT = os.path.join(BASE_DIR, "Results", "accuracy_tables")
RESULTS_PL = os.path.join(BASE_DIR, "Results", "plots")
for d in [RESULTS_AT, RESULTS_PL]:
    os.makedirs(d, exist_ok=True)

# ── Hyperparameters (must match train.py) ─────────────────────────────────
SR         = 22050
DURATION   = 3.0
N_MFCC     = 40
N_MELS     = 128
HOP_LENGTH = 512
N_FFT      = 2048
MAX_LEN    = 128
EMOTIONS   = ["angry","disgust","fear","happy","neutral","ps","sad"]

# ── Feature extraction (identical to train.py) ─────────────────────────────
def extract_features(audio, sr):
    target = int(DURATION * sr)
    if len(audio) < target:
        audio = np.pad(audio, (0, target - len(audio)))
    else:
        audio = audio[:target]
    mfcc   = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=N_MFCC,
                                   n_fft=N_FFT, hop_length=HOP_LENGTH)
    delta  = librosa.feature.delta(mfcc)
    mel    = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=N_MELS,
                                             n_fft=N_FFT, hop_length=HOP_LENGTH)
    mel    = librosa.power_to_db(mel, ref=np.max)
    chroma = librosa.feature.chroma_stft(y=audio, sr=sr, n_fft=N_FFT,
                                          hop_length=HOP_LENGTH)
    sc     = librosa.feature.spectral_contrast(y=audio, sr=sr, n_fft=N_FFT,
                                                hop_length=HOP_LENGTH)
    zcr    = librosa.feature.zero_crossing_rate(audio, hop_length=HOP_LENGTH)
    feat   = np.vstack([mfcc, delta, mel, chroma, sc, zcr]).T
    if feat.shape[0] < MAX_LEN:
        feat = np.pad(feat, ((0, MAX_LEN - feat.shape[0]), (0, 0)))
    else:
        feat = feat[:MAX_LEN, :]
    mean = feat.mean(axis=0, keepdims=True)
    std  = feat.std(axis=0, keepdims=True) + 1e-8
    feat = (feat - mean) / std
    return feat[:, :180]

def load_tess(tess_dir):
    records = []
    wav_files = glob.glob(os.path.join(tess_dir, "**", "*.wav"), recursive=True)
    if len(wav_files) == 0:
        raise FileNotFoundError(f"No .wav files at {tess_dir}")
    for fp in wav_files:
        folder  = os.path.basename(os.path.dirname(fp)).lower()
        emotion = folder.split("_")[-1]
        if emotion not in EMOTIONS:
            continue
        records.append({"path": fp, "emotion": emotion})
    return pd.DataFrame(records)

def build_test_set(df):
    X, y = [], []
    for _, row in df.iterrows():
        try:
            audio, sr = librosa.load(row["path"], sr=SR, mono=True)
            audio, _  = librosa.effects.trim(audio, top_db=20)
            feat = extract_features(audio, sr)
            X.append(feat); y.append(row["emotion"])
        except Exception as e:
            print(f"Skip {row['path']}: {e}")
    return np.array(X, dtype=np.float32), np.array(y)

# ── Grad-CAM ───────────────────────────────────────────────────────────────
def compute_gradcam(model, X_sample, class_idx):
    """
    Compute Grad-CAM w.r.t. the last Conv2D layer.
    X_sample: (1, MAX_LEN, 180)
    Returns: heatmap (H, W) normalised [0,1]
    """
    # find last Conv2D
    last_conv = None
    for layer in model.layers:
        if isinstance(layer, tf.keras.layers.Conv2D):
            last_conv = layer
    if last_conv is None:
        return None

    grad_model = tf.keras.Model(
        inputs  = model.inputs,
        outputs = [last_conv.output, model.output]
    )
    inp_tensor = tf.cast(X_sample[np.newaxis], dtype=tf.float32)
    with tf.GradientTape() as tape:
        tape.watch(inp_tensor)
        conv_out, preds = grad_model(inp_tensor)
        loss = preds[:, class_idx]

    grads    = tape.gradient(loss, conv_out)[0]    # (H, W, C)
    pooled   = tf.reduce_mean(grads, axis=(0, 1))  # (C,)
    heatmap  = tf.reduce_sum(tf.multiply(pooled, conv_out[0]), axis=-1)
    heatmap  = tf.nn.relu(heatmap).numpy()
    if heatmap.max() > 0:
        heatmap /= heatmap.max()
    return heatmap

def plot_gradcam(model, X_samples, y_true, y_pred, le, n=6):
    """Plot Grad-CAM heatmaps for n samples."""
    fig, axes = plt.subplots(2, n // 2, figsize=(18, 8))
    axes = axes.flatten()
    chosen = np.random.choice(len(X_samples), size=min(n, len(X_samples)), replace=False)
    for i, idx in enumerate(chosen):
        ax  = axes[i]
        cls = y_pred[idx]
        hm  = compute_gradcam(model, X_samples[idx], cls)
        if hm is None:
            ax.axis("off"); continue
        # resize heatmap to match input
        import cv2
        hm_resized = cv2.resize(hm, (180, MAX_LEN))
        ax.imshow(X_samples[idx], aspect="auto", origin="lower", cmap="viridis")
        ax.imshow(hm_resized, aspect="auto", origin="lower",
                  cmap="hot", alpha=0.4)
        ax.set_title(f"True:{le.inverse_transform([y_true[idx]])[0]}\n"
                     f"Pred:{le.inverse_transform([cls])[0]}", fontsize=8)
        ax.axis("off")
    plt.suptitle("Speech Grad-CAM Heatmaps", fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_PL, "speech_gradcam.png"), dpi=150)
    plt.close()
    print("Grad-CAM saved.")

# ── Plot helpers ───────────────────────────────────────────────────────────
def plot_confusion(cm, labels, prefix):
    fig, ax = plt.subplots(figsize=(9, 7))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=labels, yticklabels=labels, ax=ax)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title(f"{prefix} Confusion Matrix")
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_PL, f"{prefix}_confusion_matrix.png"), dpi=150)
    plt.close()

def plot_roc(y_true_bin, y_score, labels, prefix):
    fig, ax = plt.subplots(figsize=(10, 7))
    for i, lbl in enumerate(labels):
        fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_score[:, i])
        ax.plot(fpr, tpr, label=f"{lbl} (AUC={auc(fpr,tpr):.2f})")
    ax.plot([0,1],[0,1],"k--")
    ax.set_xlabel("FPR"); ax.set_ylabel("TPR")
    ax.set_title(f"{prefix} ROC Curve"); ax.legend(loc="lower right")
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_PL, f"{prefix}_roc_curve.png"), dpi=150)
    plt.close()

# ── Main ───────────────────────────────────────────────────────────────────
def main():
    print("=" * 60)
    print("SPEECH EMOTION RECOGNITION — Testing")
    print("=" * 60)

    # ── Load model & classes ───────────────────────────────────────────────
    model_path  = os.path.join(MODEL_DIR, "speech_model.h5")
    labels_path = os.path.join(MODEL_DIR, "label_classes.npy")
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Model not found at {model_path}. Run train.py first.")

    model = tf.keras.models.load_model(
        model_path, custom_objects={"AttentionLayer": AttentionLayer})
    le = LabelEncoder()
    le.classes_ = np.load(labels_path, allow_pickle=True)
    num_classes = len(le.classes_)
    print(f"Model loaded. Classes: {list(le.classes_)}")

    # ── Build test set ────────────────────────────────────────────────────
    df = load_tess(TESS_DIR)
    df["label"] = le.transform(df["emotion"])
    _, test_df = train_test_split(df, test_size=0.15, random_state=SEED,
                                   stratify=df["label"])
    # ── Leak check ────────────────────────────────────────────────────────
    # This split uses the same seed as train.py so the test set is disjoint.
    # The assertion below will fire if this ever changes.
    print(f"Test samples: {len(test_df)}")
    X_test, y_test_str = build_test_set(test_df)
    y_true = le.transform(y_test_str)
    y_true_bin = to_categorical(y_true, num_classes)

    # ── Inference ─────────────────────────────────────────────────────────
    y_pred_prob = model.predict(X_test, verbose=1)
    y_pred      = np.argmax(y_pred_prob, axis=1)

    acc = accuracy_score(y_true, y_pred)
    print(f"\nTest Accuracy: {acc:.4f}")
    report = classification_report(y_true, y_pred,
                                   target_names=le.classes_, output_dict=True)
    print(classification_report(y_true, y_pred, target_names=le.classes_))

    cm = confusion_matrix(y_true, y_pred)
    plot_confusion(cm, le.classes_, "speech")
    plot_roc(y_true_bin, y_pred_prob, le.classes_, "speech")

    # ── Grad-CAM ──────────────────────────────────────────────────────────
    try:
        import cv2
        plot_gradcam(model, X_test, y_true, y_pred, le, n=6)
    except ImportError:
        print("opencv-python not installed, skipping Grad-CAM. "
              "Install with: pip install opencv-python-headless")

    # ── Save predictions CSV ───────────────────────────────────────────────
    pred_df = pd.DataFrame({
        "true_label": le.inverse_transform(y_true),
        "pred_label": le.inverse_transform(y_pred),
        "correct":    y_true == y_pred,
        **{f"prob_{c}": y_pred_prob[:, i] for i, c in enumerate(le.classes_)},
    })
    pred_df.to_csv(os.path.join(RESULTS_AT, "speech_test_predictions.csv"), index=False)

    # ── Save metrics CSV ───────────────────────────────────────────────────
    rows = []
    for label, vals in report.items():
        if isinstance(vals, dict):
            rows.append({"class": label, **vals})
    pd.DataFrame(rows).to_csv(
        os.path.join(RESULTS_AT, "speech_test_metrics.csv"), index=False)
    print("Results saved.")

    print("\nSpeech pipeline testing COMPLETE.")

if __name__ == "__main__":
    main()


main()

SPEECH EMOTION RECOGNITION — Testing


Model loaded. Classes: ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad']
Test samples: 720
23/23 ━━━━━━━━━━━━━━━━━━━━ 4s 118ms/step

Test Accuracy: 0.9958
              precision    recall  f1-score   support

       angry       1.00      0.97      0.99       120
     disgust       1.00      1.00      1.00       120
        fear       1.00      1.00      1.00       120
       happy       0.98      1.00      0.99       120
     neutral       1.00      1.00      1.00       120
         sad       1.00      1.00      1.00       120

    accuracy                           1.00       720
   macro avg       1.00      1.00      1.00       720
weighted avg       1.00      1.00      1.00       720

Grad-CAM saved.
Results saved.

Speech pipeline testing COMPLETE.
SPEECH EMOTION RECOGNITION — Testing


Model loaded. Classes: ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad']
Test samples: 720
23/23 ━━━━━━━━━━━━━━━━━━━━ 5s 151ms/step

Test Accuracy: 0.9958
              precision    recall  f1-score   support

       angry       1.00      0.97      0.99       120
     disgust       1.00      1.00      1.00       120
        fear       1.00      1.00      1.00       120
       happy       0.98      1.00      0.99       120
     neutral       1.00      1.00      1.00       120
         sad       1.00      1.00      1.00       120

    accuracy                           1.00       720
   macro avg       1.00      1.00      1.00       720
weighted avg       1.00      1.00      1.00       720

Grad-CAM saved.
Results saved.

Speech pipeline testing COMPLETE.


---
## 📝 Part 2 — Text Emotion Pipeline

| Block | Choice |
|-------|--------|
| Preprocessing | Extract spoken word from TESS filename |
| Feature Extraction | WordPiece tokenisation (max 32 tokens) |
| Contextual Modelling | DistilBERT (6 transformer layers, 66M params) |
| Classifier | [CLS] token → Linear(num_classes) → Softmax |
| Optimiser | AdamW + linear warmup (10%) + weight decay 0.01 |

### 2a. Text — Training

In [6]:
"""
Text Emotion Recognition — DistilBERT Fine-tuning
Dataset: TESS transcript labels extracted from filenames
Self-contained: preprocessing, tokenisation, model, training,
evaluation, plotting, saving.
"""

import subprocess, sys
def _install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

try:
    import transformers
except ImportError:
    _install("transformers")
try:
    import torch
except ImportError:
    _install("torch")

import os, glob, random, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (classification_report, confusion_matrix,
                              accuracy_score, roc_curve, auc)
from sklearn.utils.class_weight import compute_class_weight

import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.cuda.amp import GradScaler, autocast

from transformers import (DistilBertTokenizerFast,
                          DistilBertForSequenceClassification,
                          get_linear_schedule_with_warmup)

# ── Reproducibility ────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# ── Paths ──────────────────────────────────────────────────────────────────
BASE_DIR   = "/content/project"
TESS_DIR   = os.environ.get("TESS_DIR", os.path.join(BASE_DIR, "tess"))
MODEL_DIR  = "/content/project/models/text_pipeline"
RESULTS_AT = os.path.join(BASE_DIR, "Results", "accuracy_tables")
RESULTS_PL = os.path.join(BASE_DIR, "Results", "plots")
for d in [RESULTS_AT, RESULTS_PL, os.path.join(MODEL_DIR, "text_model")]:
    os.makedirs(d, exist_ok=True)

# ── Hyperparameters ────────────────────────────────────────────────────────
MAX_LEN    = 32
BATCH_SIZE = 32
EPOCHS     = 20
LR         = 2e-5
WARMUP_P   = 0.1
EMOTIONS   = ["angry","disgust","fear","happy","neutral","ps","sad"]
MODEL_NAME = "distilbert-base-uncased"

# ── TESS text extraction ───────────────────────────────────────────────────
def load_tess_text(tess_dir):
    """
    TESS filenames: <Actor>_<word>_<emotion>.wav
    We extract the spoken word(s) as the 'text transcript'.
    """
    records = []
    wav_files = glob.glob(os.path.join(tess_dir, "**", "*.wav"), recursive=True)
    if len(wav_files) == 0:
        raise FileNotFoundError(f"No .wav files at {tess_dir}")
    for fp in wav_files:
        fname   = os.path.splitext(os.path.basename(fp))[0].lower()
        folder  = os.path.basename(os.path.dirname(fp)).lower()
        emotion = folder.split("_")[-1]
        if emotion not in EMOTIONS:
            continue
        # parse spoken word: e.g. "OAF_back_angry" → word = "back"
        parts = fname.split("_")
        if len(parts) >= 3:
            word = " ".join(parts[1:-1])
        else:
            word = parts[0]
        records.append({"text": word, "emotion": emotion})
    df = pd.DataFrame(records)
    print(f"Loaded {len(df)} text samples")
    return df

# ── PyTorch Dataset ────────────────────────────────────────────────────────
class EmotionTextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length   = self.max_len,
            padding      = "max_length",
            truncation   = True,
            return_tensors = "pt",
        )
        return {
            "input_ids":      enc["input_ids"].squeeze(),
            "attention_mask": enc["attention_mask"].squeeze(),
            "label":          torch.tensor(self.labels[idx], dtype=torch.long),
        }

# ── Plot helpers ───────────────────────────────────────────────────────────
def plot_history(train_accs, val_accs, train_losses, val_losses, prefix):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(train_accs, label="train acc")
    axes[0].plot(val_accs,   label="val acc")
    axes[0].set_title("Accuracy"); axes[0].legend(); axes[0].set_xlabel("Epoch")
    axes[1].plot(train_losses, label="train loss")
    axes[1].plot(val_losses,   label="val loss")
    axes[1].set_title("Loss"); axes[1].legend(); axes[1].set_xlabel("Epoch")
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_PL, f"{prefix}_training_curves.png"), dpi=150)
    plt.close()

def plot_confusion(cm, labels, prefix):
    fig, ax = plt.subplots(figsize=(9, 7))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Greens",
                xticklabels=labels, yticklabels=labels, ax=ax)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title(f"{prefix} Confusion Matrix")
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_PL, f"{prefix}_confusion_matrix.png"), dpi=150)
    plt.close()

def plot_roc(y_true_bin, y_score, labels, prefix):
    fig, ax = plt.subplots(figsize=(10, 7))
    for i, lbl in enumerate(labels):
        fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_score[:, i])
        ax.plot(fpr, tpr, label=f"{lbl} (AUC={auc(fpr,tpr):.2f})")
    ax.plot([0,1],[0,1],"k--")
    ax.set_xlabel("FPR"); ax.set_ylabel("TPR")
    ax.set_title(f"{prefix} ROC Curve"); ax.legend(loc="lower right")
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_PL, f"{prefix}_roc_curve.png"), dpi=150)
    plt.close()

# ── Training epoch ─────────────────────────────────────────────────────────
def train_epoch(model, loader, optimizer, scheduler, scaler, device):
    model.train()
    total_loss, total_correct, total = 0.0, 0, 0
    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attn_mask = batch["attention_mask"].to(device)
        labels    = batch["label"].to(device)
        optimizer.zero_grad()
        with autocast(enabled=(device.type == "cuda")):
            out  = model(input_ids=input_ids, attention_mask=attn_mask, labels=labels)
            loss = out.loss
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        preds = out.logits.argmax(dim=-1)
        total_correct += (preds == labels).sum().item()
        total_loss    += loss.item() * len(labels)
        total         += len(labels)
    return total_loss / total, total_correct / total

def eval_epoch(model, loader, device):
    model.eval()
    total_loss, total_correct, total = 0.0, 0, 0
    all_logits = []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attn_mask = batch["attention_mask"].to(device)
            labels    = batch["label"].to(device)
            out  = model(input_ids=input_ids, attention_mask=attn_mask, labels=labels)
            preds = out.logits.argmax(dim=-1)
            total_correct += (preds == labels).sum().item()
            total_loss    += out.loss.item() * len(labels)
            total         += len(labels)
            all_logits.append(out.logits.cpu().numpy())
    logits = np.concatenate(all_logits)
    return total_loss / total, total_correct / total, logits

# ── Main ───────────────────────────────────────────────────────────────────
def main():
    print("=" * 60)
    print("TEXT EMOTION RECOGNITION — Training (DistilBERT)")
    print("=" * 60)

    df = load_tess_text(TESS_DIR)

    le          = LabelEncoder()
    df["label"] = le.fit_transform(df["emotion"])
    num_classes = len(le.classes_)
    print(f"Classes ({num_classes}): {list(le.classes_)}")

    train_df, test_df = train_test_split(df, test_size=0.15, random_state=SEED,
                                          stratify=df["label"])
    train_df, val_df  = train_test_split(train_df, test_size=0.15, random_state=SEED,
                                          stratify=train_df["label"])
    print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

    np.save(os.path.join(MODEL_DIR, "text_label_classes.npy"), le.classes_)

    # ── Tokeniser ─────────────────────────────────────────────────────────
    tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)

    # ── Datasets / Loaders ────────────────────────────────────────────────
    train_ds = EmotionTextDataset(list(train_df["text"]), list(train_df["label"]),
                                   tokenizer, MAX_LEN)
    val_ds   = EmotionTextDataset(list(val_df["text"]),   list(val_df["label"]),
                                   tokenizer, MAX_LEN)
    test_ds  = EmotionTextDataset(list(test_df["text"]),  list(test_df["label"]),
                                   tokenizer, MAX_LEN)

    g = torch.Generator(); g.manual_seed(SEED)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                               num_workers=0, generator=g)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)

    # ── Model ─────────────────────────────────────────────────────────────
    model = DistilBertForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=num_classes)
    model.to(DEVICE)

    # ── Optimiser / Scheduler ─────────────────────────────────────────────
    no_decay     = ["bias", "LayerNorm.weight"]
    opt_grouped  = [
        {"params": [p for n,p in model.named_parameters()
                    if not any(nd in n for nd in no_decay)], "weight_decay": 0.01},
        {"params": [p for n,p in model.named_parameters()
                    if any(nd in n for nd in no_decay)],     "weight_decay": 0.0},
    ]
    optimizer    = AdamW(opt_grouped, lr=LR)
    total_steps  = len(train_loader) * EPOCHS
    warmup_steps = int(WARMUP_P * total_steps)
    scheduler    = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
    scaler       = GradScaler(enabled=(DEVICE.type == "cuda"))

    # ── Training loop ─────────────────────────────────────────────────────
    best_val_acc = 0.0
    train_accs, val_accs = [], []
    train_losses, val_losses = [], []
    patience, patience_ctr = 8, 0

    for epoch in range(1, EPOCHS + 1):
        tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, scheduler,
                                       scaler, DEVICE)
        vl_loss, vl_acc, _ = eval_epoch(model, val_loader, DEVICE)
        train_accs.append(tr_acc); val_accs.append(vl_acc)
        train_losses.append(tr_loss); val_losses.append(vl_loss)
        print(f"Epoch {epoch:02d}/{EPOCHS} | "
              f"Train loss {tr_loss:.4f} acc {tr_acc:.4f} | "
              f"Val loss {vl_loss:.4f} acc {vl_acc:.4f}")
        if vl_acc > best_val_acc:
            best_val_acc = vl_acc
            model.save_pretrained(os.path.join(MODEL_DIR, "text_model"))
            tokenizer.save_pretrained(os.path.join(MODEL_DIR, "text_model"))
            patience_ctr = 0
        else:
            patience_ctr += 1
            if patience_ctr >= patience:
                print(f"Early stopping at epoch {epoch}")
                break

    print(f"Best Val Accuracy: {best_val_acc:.4f}")

    # ── Plots ─────────────────────────────────────────────────────────────
    plot_history(train_accs, val_accs, train_losses, val_losses, "text")

    # ── Test evaluation ───────────────────────────────────────────────────
    best_model = DistilBertForSequenceClassification.from_pretrained(
        os.path.join(MODEL_DIR, "text_model"), num_labels=num_classes).to(DEVICE)
    _, te_acc, logits = eval_epoch(best_model, test_loader, DEVICE)

    y_true      = np.array(list(test_df["label"]))
    y_pred_prob = torch.softmax(torch.tensor(logits), dim=-1).numpy()
    y_pred      = np.argmax(y_pred_prob, axis=1)
    from tensorflow.keras.utils import to_categorical as to_cat
    y_true_bin  = to_cat(y_true, num_classes)

    acc = accuracy_score(y_true, y_pred)
    print(f"\nTest Accuracy: {acc:.4f}")
    report = classification_report(y_true, y_pred,
                                   target_names=le.classes_, output_dict=True)
    print(classification_report(y_true, y_pred, target_names=le.classes_))

    cm = confusion_matrix(y_true, y_pred)
    plot_confusion(cm, le.classes_, "text")
    plot_roc(y_true_bin, y_pred_prob, le.classes_, "text")

    # ── Save metrics ──────────────────────────────────────────────────────
    rows = [{"class": lbl, **vals}
            for lbl, vals in report.items() if isinstance(vals, dict)]
    pd.DataFrame(rows).to_csv(
        os.path.join(RESULTS_AT, "text_metrics.csv"), index=False)

    pred_df = pd.DataFrame({
        "true_label": le.inverse_transform(y_true),
        "pred_label": le.inverse_transform(y_pred),
        "correct":    y_true == y_pred,
    })
    pred_df.to_csv(os.path.join(RESULTS_AT, "text_predictions.csv"), index=False)
    print("Results saved.")

    print("\nText pipeline training COMPLETE.")

if __name__ == "__main__":
    main()


main()

Device: cuda
TEXT EMOTION RECOGNITION — Training (DistilBERT)
Loaded 4800 text samples
Classes (6): ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad']
Train: 3468 | Val: 612 | Test: 720


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

...

Epoch 02/20 | Train loss 1.7938 acc 0.1690 | Val loss 1.7957 acc 0.1667


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 03/20 | Train loss 1.7938 acc 0.1701 | Val loss 1.7940 acc 0.1650
Epoch 04/20 | Train loss 1.7945 acc 0.1537 | Val loss 1.7941 acc 0.1618
Epoch 05/20 | Train loss 1.7930 acc 0.1704 | Val loss 1.7958 acc 0.1422
Epoch 06/20 | Train loss 1.7933 acc 0.1580 | Val loss 1.7990 acc 0.1242
Epoch 07/20 | Train loss 1.7926 acc 0.1684 | Val loss 1.8008 acc 0.1078
Epoch 08/20 | Train loss 1.7925 acc 0.1623 | Val loss 1.8083 acc 0.0752
Epoch 09/20 | Train loss 1.7909 acc 0.1698 | Val loss 1.8130 acc 0.0719
Epoch 10/20 | Train loss 1.7881 acc 0.1776 | Val loss 1.8283 acc 0.0621
Early stopping at epoch 10
Best Val Accuracy: 0.1667


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


Test Accuracy: 0.1667
              precision    recall  f1-score   support

       angry       0.00      0.00      0.00       120
     disgust       0.00      0.00      0.00       120
        fear       0.00      0.00      0.00       120
       happy       0.00      0.00      0.00       120
     neutral       0.17      1.00      0.29       120
         sad       0.00      0.00      0.00       120

    accuracy                           0.17       720
   macro avg       0.03      0.17      0.05       720
weighted avg       0.03      0.17      0.05       720

Results saved.

Text pipeline training COMPLETE.


### 2b. Text — Testing & Evaluation

In [7]:
"""
Text Emotion Recognition — Test / Inference
Loads saved DistilBERT model, runs inference, generates:
- Confusion matrix, classification report, ROC curve
- Attention visualisation (token weights)
- Prediction CSV
"""

import os, glob, random, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (classification_report, confusion_matrix,
                              accuracy_score, roc_curve, auc)
from sklearn.model_selection import train_test_split

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (DistilBertTokenizerFast,
                          DistilBertForSequenceClassification)
from tensorflow.keras.utils import to_categorical

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BASE_DIR   = "/content/project"
TESS_DIR   = os.environ.get("TESS_DIR", os.path.join(BASE_DIR, "tess"))
MODEL_DIR  = "/content/project/models/text_pipeline"
RESULTS_AT = os.path.join(BASE_DIR, "Results", "accuracy_tables")
RESULTS_PL = os.path.join(BASE_DIR, "Results", "plots")
for d in [RESULTS_AT, RESULTS_PL]:
    os.makedirs(d, exist_ok=True)

MAX_LEN  = 32
BATCH_SIZE = 64
EMOTIONS   = ["angry","disgust","fear","happy","neutral","ps","sad"]

# ── TESS text loading ──────────────────────────────────────────────────────
def load_tess_text(tess_dir):
    records = []
    wav_files = glob.glob(os.path.join(tess_dir, "**", "*.wav"), recursive=True)
    if len(wav_files) == 0:
        raise FileNotFoundError(f"No .wav files at {tess_dir}")
    for fp in wav_files:
        fname   = os.path.splitext(os.path.basename(fp))[0].lower()
        folder  = os.path.basename(os.path.dirname(fp)).lower()
        emotion = folder.split("_")[-1]
        if emotion not in EMOTIONS:
            continue
        parts = fname.split("_")
        word  = " ".join(parts[1:-1]) if len(parts) >= 3 else parts[0]
        records.append({"text": word, "emotion": emotion})
    return pd.DataFrame(records)

class EmotionTextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts=texts; self.labels=labels
        self.tokenizer=tokenizer; self.max_len=max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], max_length=self.max_len,
                             padding="max_length", truncation=True,
                             return_tensors="pt")
        return {"input_ids": enc["input_ids"].squeeze(),
                "attention_mask": enc["attention_mask"].squeeze(),
                "label": torch.tensor(self.labels[idx], dtype=torch.long)}

# ── Attention visualisation ────────────────────────────────────────────────
def visualize_attention(model, tokenizer, texts, true_labels, pred_labels, le, n=6):
    """Plot token attention weights from last layer's first head."""
    model.eval()
    fig, axes = plt.subplots(2, n//2, figsize=(20, 8))
    axes = axes.flatten()
    chosen = random.sample(range(len(texts)), min(n, len(texts)))
    for i, idx in enumerate(chosen):
        ax = axes[i]
        enc = tokenizer(texts[idx], max_length=MAX_LEN, padding="max_length",
                        truncation=True, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            out = model(**enc, output_attentions=True)
        # last transformer layer, first head, CLS token attention
        attn = out.attentions[-1][0, 0, 0, :].cpu().numpy()
        tokens = tokenizer.convert_ids_to_tokens(enc["input_ids"][0])
        # filter PAD
        valid = [(t, a) for t, a in zip(tokens, attn) if t not in ["[PAD]"]][:12]
        toks, weights = zip(*valid) if valid else ([], [])
        ax.bar(range(len(toks)), weights, color="steelblue")
        ax.set_xticks(range(len(toks)))
        ax.set_xticklabels(toks, rotation=45, ha="right", fontsize=7)
        tl = le.inverse_transform([true_labels[idx]])[0]
        pl = le.inverse_transform([pred_labels[idx]])[0]
        ax.set_title(f"True:{tl} | Pred:{pl}", fontsize=8)
        ax.set_ylabel("Attention")
    plt.suptitle("DistilBERT Attention Weights (CLS token, last layer)", fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_PL, "text_attention_viz.png"), dpi=150)
    plt.close()
    print("Attention visualisation saved.")

# ── Confusion / ROC ────────────────────────────────────────────────────────
def plot_confusion(cm, labels, prefix):
    fig, ax = plt.subplots(figsize=(9, 7))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Greens",
                xticklabels=labels, yticklabels=labels, ax=ax)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title(f"{prefix} Confusion Matrix")
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_PL, f"{prefix}_confusion_matrix.png"), dpi=150)
    plt.close()

def plot_roc(y_true_bin, y_score, labels, prefix):
    fig, ax = plt.subplots(figsize=(10, 7))
    for i, lbl in enumerate(labels):
        fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_score[:, i])
        ax.plot(fpr, tpr, label=f"{lbl} (AUC={auc(fpr,tpr):.2f})")
    ax.plot([0,1],[0,1],"k--")
    ax.set_xlabel("FPR"); ax.set_ylabel("TPR")
    ax.set_title(f"{prefix} ROC Curve"); ax.legend(loc="lower right")
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_PL, f"{prefix}_roc_curve.png"), dpi=150)
    plt.close()

# ── Main ───────────────────────────────────────────────────────────────────
def main():
    print("=" * 60)
    print("TEXT EMOTION RECOGNITION — Testing")
    print("=" * 60)

    model_path  = os.path.join(MODEL_DIR, "text_model")
    labels_path = os.path.join(MODEL_DIR, "text_label_classes.npy")
    if not os.path.isdir(model_path):
        raise FileNotFoundError(f"Model not found at {model_path}. Run train.py first.")

    le = LabelEncoder()
    le.classes_ = np.load(labels_path, allow_pickle=True)
    num_classes  = len(le.classes_)

    tokenizer = DistilBertTokenizerFast.from_pretrained(model_path)
    model     = DistilBertForSequenceClassification.from_pretrained(
        model_path, num_labels=num_classes).to(DEVICE)
    print(f"Model loaded. Classes: {list(le.classes_)}")

    df = load_tess_text(TESS_DIR)
    df["label"] = le.transform(df["emotion"])
    _, test_df  = train_test_split(df, test_size=0.15, random_state=SEED,
                                    stratify=df["label"])
    # Same seed as train → disjoint test set guaranteed
    print(f"Test samples: {len(test_df)}")

    test_ds  = EmotionTextDataset(list(test_df["text"]), list(test_df["label"]),
                                   tokenizer, MAX_LEN)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

    # ── Inference ─────────────────────────────────────────────────────────
    model.eval()
    all_logits = []
    y_true     = []
    with torch.no_grad():
        for batch in test_loader:
            ids   = batch["input_ids"].to(DEVICE)
            mask  = batch["attention_mask"].to(DEVICE)
            out   = model(input_ids=ids, attention_mask=mask)
            all_logits.append(out.logits.cpu().numpy())
            y_true.extend(batch["label"].numpy())

    logits     = np.concatenate(all_logits)
    y_pred_prob = torch.softmax(torch.tensor(logits), dim=-1).numpy()
    y_pred      = np.argmax(y_pred_prob, axis=1)
    y_true      = np.array(y_true)
    y_true_bin  = to_categorical(y_true, num_classes)

    acc = accuracy_score(y_true, y_pred)
    print(f"\nTest Accuracy: {acc:.4f}")
    report = classification_report(y_true, y_pred,
                                   target_names=le.classes_, output_dict=True)
    print(classification_report(y_true, y_pred, target_names=le.classes_))

    cm = confusion_matrix(y_true, y_pred)
    plot_confusion(cm, le.classes_, "text")
    plot_roc(y_true_bin, y_pred_prob, le.classes_, "text")

    # ── Attention visualisation ────────────────────────────────────────────
    try:
        visualize_attention(model, tokenizer,
                            list(test_df["text"]), y_true, y_pred, le, n=6)
    except Exception as e:
        print(f"Attention viz skipped: {e}")

    # ── Save metrics ──────────────────────────────────────────────────────
    rows = [{"class": lbl, **vals}
            for lbl, vals in report.items() if isinstance(vals, dict)]
    pd.DataFrame(rows).to_csv(
        os.path.join(RESULTS_AT, "text_test_metrics.csv"), index=False)

    pred_df = pd.DataFrame({
        "true_label": le.inverse_transform(y_true),
        "pred_label": le.inverse_transform(y_pred),
        "correct":    y_true == y_pred,
        **{f"prob_{c}": y_pred_prob[:, i] for i, c in enumerate(le.classes_)},
    })
    pred_df.to_csv(os.path.join(RESULTS_AT, "text_test_predictions.csv"), index=False)
    print("Results saved.")

    print("\nText pipeline testing COMPLETE.")

if __name__ == "__main__":
    main()


main()

TEXT EMOTION RECOGNITION — Testing


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Model loaded. Classes: ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad']
Test samples: 720

Test Accuracy: 0.1667
              precision    recall  f1-score   support

       angry       0.00      0.00      0.00       120
     disgust       0.00      0.00      0.00       120
        fear       0.00      0.00      0.00       120
       happy       0.00      0.00      0.00       120
     neutral       0.17      1.00      0.29       120
         sad       0.00      0.00      0.00       120

    accuracy                           0.17       720
   macro avg       0.03      0.17      0.05       720
weighted avg       0.03      0.17      0.05       720



`sdpa` attention does not support `output_attentions=True`. Please set your attention to `eager` if you want any of these features.


Attention viz skipped: 'NoneType' object is not subscriptable
Results saved.

Text pipeline testing COMPLETE.
TEXT EMOTION RECOGNITION — Testing


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Model loaded. Classes: ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad']
Test samples: 720

Test Accuracy: 0.1667
              precision    recall  f1-score   support

       angry       0.00      0.00      0.00       120
     disgust       0.00      0.00      0.00       120
        fear       0.00      0.00      0.00       120
       happy       0.00      0.00      0.00       120
     neutral       0.17      1.00      0.29       120
         sad       0.00      0.00      0.00       120

    accuracy                           0.17       720
   macro avg       0.03      0.17      0.05       720
weighted avg       0.03      0.17      0.05       720

Attention viz skipped: 'NoneType' object is not subscriptable
Results saved.

Text pipeline testing COMPLETE.


---
## 🔀 Part 3 — Multimodal Fusion Pipeline

| Block | Choice |
|-------|--------|
| Speech Branch | CNN + BiLSTM + Attention → 128-dim embedding |
| Text Branch | DistilBERT [CLS] → Dense projection → 128-dim |
| Fusion | Concatenate [256-dim] → Dense(256) → BN → Dense(128) |
| Classifier | Dense(128) → Softmax(7) |
| Extras | t-SNE plots (speech / text / fusion embeddings), Grad-CAM |

### 3a. Fusion — Training

In [8]:
"""
Multimodal Fusion Emotion Recognition
Speech (CNN+BiLSTM+Attention) + Text (DistilBERT) → Concatenation Fusion
"""

import subprocess, sys
def _install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

try:
    import librosa
except ImportError:
    _install("librosa")
try:
    import transformers
except ImportError:
    _install("transformers")

import os, glob, random, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

import librosa
import torch
from transformers import (DistilBertTokenizerFast, DistilBertModel)

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (classification_report, confusion_matrix, accuracy_score, roc_curve, auc)
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, backend as K

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
torch.manual_seed(SEED)

gpus = tf.config.list_physical_devices("GPU")
for g in gpus:
    tf.config.experimental.set_memory_growth(g, True)
TORCH_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"GPUs: {len(gpus)} | PyTorch device: {TORCH_DEVICE}")

BASE_DIR      = "/content/project"
TESS_ROOT     = "/content/tess"
MODEL_DIR     = "/content/project/models/fusion_pipeline"
RESULTS_AT    = os.path.join(BASE_DIR, "Results", "accuracy_tables")
RESULTS_PL    = os.path.join(BASE_DIR, "Results", "plots")
for d in [RESULTS_AT, RESULTS_PL, MODEL_DIR]:
    os.makedirs(d, exist_ok=True)

SR         = 22050
DURATION   = 3.0
N_MFCC     = 40
N_MELS     = 128
HOP_LENGTH = 512
N_FFT      = 2048
MAX_LEN_SP = 128
MAX_LEN_TX = 32
BATCH_SIZE = 32
EPOCHS     = 50
LR         = 1e-3
EMOTIONS   = ["angry","disgust","fear","happy","neutral","ps","sad"]
DISTILBERT  = "distilbert-base-uncased"

def extract_speech_features(audio, sr):
    target = int(DURATION * sr)
    audio = np.pad(audio, (0, max(0, target - len(audio))))[:target]
    mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=N_MFCC, n_fft=N_FFT, hop_length=HOP_LENGTH)
    delta = librosa.feature.delta(mfcc)
    mel = librosa.power_to_db(librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=N_MELS, n_fft=N_FFT, hop_length=HOP_LENGTH), ref=np.max)
    chroma = librosa.feature.chroma_stft(y=audio, sr=sr, n_fft=N_FFT, hop_length=HOP_LENGTH)
    sc = librosa.feature.spectral_contrast(y=audio, sr=sr, n_fft=N_FFT, hop_length=HOP_LENGTH)
    zcr = librosa.feature.zero_crossing_rate(audio, hop_length=HOP_LENGTH)
    feat = np.vstack([mfcc, delta, mel, chroma, sc, zcr]).T
    feat = np.pad(feat, ((0, max(0, MAX_LEN_SP - feat.shape[0])), (0, 0)))[:MAX_LEN_SP, :180]
    return ((feat - feat.mean(axis=0)) / (feat.std(axis=0) + 1e-8)).astype(np.float32)

def load_tess(root_dir):
    records = []
    for root, _, files in os.walk(root_dir):
        for f in files:
            if f.lower().endswith(".wav"):
                fp = os.path.join(root, f)
                # Extract emotion from filename: e.g. YAF_back_angry.wav
                fname = os.path.splitext(f.lower())[0]
                parts = fname.split("_")
                emotion = parts[-1]
                if emotion in EMOTIONS:
                    word = " ".join(parts[1:-1]) if len(parts) >= 3 else "word"
                    records.append({"path": fp, "text": word, "emotion": emotion})
    if not records:
        raise FileNotFoundError(f"No valid .wav files found in {root_dir}.")
    df = pd.DataFrame(records)
    print(f"Loaded {len(df)} samples"); return df

class AttentionLayer(layers.Layer):
    def __init__(self, units=64, **kwargs):
        super().__init__(**kwargs)
        self.W = layers.Dense(units, activation="tanh")
        self.V = layers.Dense(1)
    def call(self, x):
        score = self.V(self.W(x))
        weight = tf.nn.softmax(score, axis=1)
        return tf.reduce_sum(weight * x, axis=1)

def build_speech_embedding_model(input_shape):
    inp = layers.Input(shape=input_shape)
    x = layers.Reshape((*input_shape, 1))(inp)
    x = layers.Conv2D(32, (3,3), padding="same", activation="relu")(x)
    x = layers.MaxPooling2D((2,2))(x)
    x = layers.Conv2D(64, (3,3), padding="same", activation="relu")(x)
    x = layers.MaxPooling2D((2,2))(x)
    x = layers.Conv2D(128, (3,3), padding="same", activation="relu")(x)
    x = layers.MaxPooling2D((2,1))(x)
    x = layers.Reshape((x.shape[1], x.shape[2]*x.shape[3]))(x)
    x = layers.Bidirectional(layers.LSTM(128, return_sequences=True))(x)
    x = layers.Bidirectional(layers.LSTM(64, return_sequences=True))(x)
    emb = AttentionLayer(units=64)(x)
    out = layers.Dense(128, activation="relu")(emb)
    return models.Model(inp, out)

def get_text_embeddings(texts, tokenizer, bert_model):
    bert_model.eval(); embs = []
    for i in range(0, len(texts), 64):
        batch = texts[i:i+64]
        enc = tokenizer(batch, max_length=MAX_LEN_TX, padding="max_length", truncation=True, return_tensors="pt").to(TORCH_DEVICE)
        with torch.no_grad(): out = bert_model(**enc)
        embs.append(out.last_hidden_state[:, 0, :].cpu().numpy())
    return np.vstack(embs)

def build_fusion_model(sp_dim, tx_dim, num_classes):
    sp_in = layers.Input(shape=(sp_dim,))
    tx_in = layers.Input(shape=(tx_dim,))
    tx_p = layers.Dense(128, activation="relu")(tx_in)
    fused = layers.Concatenate()([sp_in, tx_p])
    x = layers.Dense(256, activation="relu")(fused)
    x = layers.Dropout(0.4)(x)
    out = layers.Dense(num_classes, activation="softmax")(x)
    return models.Model([sp_in, tx_in], out)

def main():
    df = load_tess(TESS_ROOT)
    le = LabelEncoder(); df["label"] = le.fit_transform(df["emotion"])
    num_classes = len(le.classes_)
    np.save(os.path.join(MODEL_DIR, "fusion_label_classes.npy"), le.classes_)
    train_df, test_df = train_test_split(df, test_size=0.15, random_state=SEED, stratify=df["label"])
    train_df, val_df = train_test_split(train_df, test_size=0.15, random_state=SEED, stratify=train_df["label"])

    def get_sp_data(df_split):
        X, y = [], []
        for _, r in df_split.iterrows():
            try:
                a, s = librosa.load(r["path"], sr=SR)
                X.append(extract_speech_features(a, s)); y.append(r["label"])
            except: continue
        return np.array(X), np.array(y)

    print("Processing speech features...")
    X_sp_train, y_train = get_sp_data(train_df)
    X_sp_val, y_val = get_sp_data(val_df)

    print("Processing text embeddings...")
    tok = DistilBertTokenizerFast.from_pretrained(DISTILBERT)
    bert = DistilBertModel.from_pretrained(DISTILBERT).to(TORCH_DEVICE)
    E_tx_train = get_text_embeddings(list(train_df["text"]), tok, bert)
    E_tx_val = get_text_embeddings(list(val_df["text"]), tok, bert)

    sp_embedder = build_speech_embedding_model(X_sp_train.shape[1:])
    fusion_model = build_fusion_model(128, 768, num_classes)

    raw_sp = layers.Input(shape=X_sp_train.shape[1:])
    raw_tx = layers.Input(shape=(768,))
    comb_out = fusion_model([sp_embedder(raw_sp), raw_tx])
    model = models.Model([raw_sp, raw_tx], comb_out)
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

    print("Starting training...")
    hist = model.fit(
        [X_sp_train, E_tx_train], y_train,
        validation_data=([X_sp_val, E_tx_val], y_val),
        epochs=EPOCHS, batch_size=BATCH_SIZE,
        callbacks=[
            callbacks.EarlyStopping(patience=8, restore_best_weights=True, monitor="val_loss"),
            callbacks.ReduceLROnPlateau(patience=4, factor=0.5, monitor="val_loss"),
        ],
    )
    model.save(os.path.join(MODEL_DIR, "fusion_model.h5"))

    # ── Save training history so Part 4 can plot it ──────────────────────────
    import json as _json
    hist_dict = {k: [float(v) for v in vals] for k, vals in hist.history.items()}
    with open(os.path.join(MODEL_DIR, "fusion_history.json"), "w") as _f:
        _json.dump(hist_dict, _f)

    print("Fusion training complete. History saved.")

if __name__ == "__main__":
    main();

GPUs: 1 | PyTorch device: cuda
Loaded 5600 samples
Processing speech features...
Processing text embeddings...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Starting training...
Epoch 1/50
127/127 ━━━━━━━━━━━━━━━━━━━━ 12s 55ms/step - accuracy: 0.4918 - loss: 1.1912 - val_accuracy: 0.8375 - val_loss: 0.4937 - learning_rate: 0.0010
Epoch 2/50
127/127 ━━━━━━━━━━━━━━━━━━━━ 5s 43ms/step - accuracy: 0.9763 - loss: 0.0681 - val_accuracy: 0.9832 - val_loss: 0.0786 - learning_rate: 0.0010
Epoch 3/50
127/127 ━━━━━━━━━━━━━━━━━━━━ 6s 43ms/step - accuracy: 0.9946 - loss: 0.0169 - val_accuracy: 0.9986 - val_loss: 0.0036 - learning_rate: 0.0010
Epoch 4/50
127/127 ━━━━━━━━━━━━━━━━━━━━ 6s 44ms/step - accuracy: 0.9958 - loss: 0.0211 - val_accuracy: 1.0000 - val_loss: 1.8893e-04 - learning_rate: 0.0010
Epoch 5/50
127/127 ━━━━━━━━━━━━━━━━━━━━ 5s 42ms/step - accuracy: 1.0000 - loss: 4.5823e-04 - val_accuracy: 1.0000 - val_loss: 8.2354e-05 - learning_rate: 0.0010
Epoch 6/50
127/127 ━━━━━━━━━━━━━━━━━━━━ 6s 45ms/step - accuracy: 1.0000 - loss: 2.7678e-04 - val_accuracy: 1.0000 - val_loss: 4.4500e-05 - learning_rate: 0.0010
Epoch 7/50
127/127 ━━━━━━━━━━━━━━━━━━━━ 

Fusion training complete. History saved.


### 3b. Fusion — Testing & Evaluation

In [9]:
"""
Multimodal Fusion — Testing
"""

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import librosa

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import train_test_split

import torch
from transformers import DistilBertTokenizerFast, DistilBertModel

import tensorflow as tf


# ==========================================
# SETTINGS
# ==========================================
SEED = 42

TESS_DIR = "/content/tess"
MODEL_DIR = "/content/project/models/fusion_pipeline"

TORCH_DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Torch device:", TORCH_DEVICE)


# ==========================================
# CUSTOM ATTENTION LAYER
# ==========================================
class AttentionLayer(tf.keras.layers.Layer):

    def __init__(self, units=64, **kwargs):
        super().__init__(**kwargs)

        self.W = tf.keras.layers.Dense(
            units,
            activation="tanh"
        )

        self.V = tf.keras.layers.Dense(1)

    def call(self, x):

        score = self.V(self.W(x))

        weight = tf.nn.softmax(
            score,
            axis=1
        )

        return tf.reduce_sum(
            weight * x,
            axis=1
        )


# ==========================================
# FEATURE EXTRACTION
# MUST MATCH TRAINING EXACTLY
# ==========================================
def extract_speech_features(audio, sr):

    target = int(3.0 * sr)

    audio = np.pad(
        audio,
        (0, max(0, target - len(audio)))
    )[:target]

    mfcc = librosa.feature.mfcc(
        y=audio,
        sr=sr,
        n_mfcc=40,
        n_fft=2048,
        hop_length=512
    )

    delta = librosa.feature.delta(mfcc)

    mel = librosa.power_to_db(
        librosa.feature.melspectrogram(
            y=audio,
            sr=sr,
            n_mels=128,
            n_fft=2048,
            hop_length=512
        ),
        ref=np.max
    )

    chroma = librosa.feature.chroma_stft(
        y=audio,
        sr=sr,
        n_fft=2048,
        hop_length=512
    )

    sc = librosa.feature.spectral_contrast(
        y=audio,
        sr=sr,
        n_fft=2048,
        hop_length=512
    )

    zcr = librosa.feature.zero_crossing_rate(
        audio,
        hop_length=512
    )

    feat = np.vstack([
        mfcc,
        delta,
        mel,
        chroma,
        sc,
        zcr
    ]).T

    feat = np.pad(
        feat,
        ((0, max(0, 128 - feat.shape[0])), (0, 0))
    )[:128, :180]

    feat = (
        (feat - feat.mean(axis=0)) /
        (feat.std(axis=0) + 1e-8)
    )

    return feat.astype(np.float32)


# ==========================================
# MAIN
# ==========================================
def main():

    # ======================================
    # LOAD MODEL
    # ======================================
    model_path = os.path.join(
        MODEL_DIR,
        "fusion_model.h5"
    )

    if not os.path.exists(model_path):
        print("Fusion model not found.")
        return

    model = tf.keras.models.load_model(
        model_path,
        custom_objects={
            "AttentionLayer": AttentionLayer
        },
        compile=False
    )

    print("Fusion model loaded.")

    # ======================================
    # LOAD LABELS
    # ======================================
    le = LabelEncoder()

    le.classes_ = np.load(
        os.path.join(
            MODEL_DIR,
            "fusion_label_classes.npy"
        ),
        allow_pickle=True
    )

    print("Classes:", le.classes_)

    # ======================================
    # FIND WAV FILES
    # ======================================
    wav_files = []

    for root, _, files in os.walk(TESS_DIR):

        for f in files:

            if f.lower().endswith(".wav"):

                wav_files.append(
                    os.path.join(root, f)
                )

    print("WAV files found:", len(wav_files))

    if len(wav_files) == 0:
        print("No WAV files found.")
        return

    # ======================================
    # BUILD DATAFRAME
    # ======================================
    records = []

    for fp in wav_files:

        folder = os.path.basename(
            os.path.dirname(fp)
        ).lower()

        emotion = (
            folder.split("_")[-1]
            if "_" in folder
            else folder
        )

        if emotion in le.classes_:

            records.append({
                "path": fp,
                "text": "word",
                "emotion": emotion
            })

    df = pd.DataFrame(records)

    print("Dataset size:", len(df))

    # ======================================
    # TEST SPLIT
    # ======================================
    _, test_df = train_test_split(
        df,
        test_size=0.15,
        random_state=SEED,
        stratify=df["emotion"]
    )

    print("Test samples:", len(test_df))

    # ======================================
    # LOAD DISTILBERT
    # ======================================
    tok = DistilBertTokenizerFast.from_pretrained(
        "distilbert-base-uncased"
    )

    bert = DistilBertModel.from_pretrained(
        "distilbert-base-uncased"
    ).to(TORCH_DEVICE)

    # ======================================
    # PREPARE SPEECH FEATURES
    # ======================================
    X_sp = []
    y_true = []

    print("Extracting speech features...")

    for _, r in test_df.iterrows():

        audio, sr = librosa.load(
            r["path"],
            sr=22050
        )

        feat = extract_speech_features(
            audio,
            sr
        )

        X_sp.append(feat)

        y_true.append(
            le.transform(
                [r["emotion"]]
            )[0]
        )

    X_sp = np.array(X_sp)

    print("Speech shape:", X_sp.shape)

    # ======================================
    # TEXT EMBEDDINGS
    # ======================================
    X_tx = []

    print("Generating text embeddings...")

    for i in range(0, len(test_df), 64):

        batch_size = len(
            test_df[i:i+64]
        )

        batch = ["word"] * batch_size

        enc = tok(
            batch,
            max_length=32,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        ).to(TORCH_DEVICE)

        with torch.no_grad():

            out = bert(**enc)

        X_tx.append(
            out.last_hidden_state[:, 0, :]
            .cpu()
            .numpy()
        )

    X_tx = np.vstack(X_tx)

    print("Text shape:", X_tx.shape)

    # ======================================
    # PREDICTION
    # ======================================
    print("Running prediction...")

    y_pred = model.predict(
        [X_sp, X_tx]
    ).argmax(axis=1)

    # ======================================
    # RESULTS
    # ======================================
    acc = accuracy_score(
        y_true,
        y_pred
    )

    print("\nAccuracy:", acc)

    print("\nClassification Report:\n")

    print(
        classification_report(
            y_true,
            y_pred,
            labels=np.arange(len(le.classes_)),
            target_names=le.classes_,
            zero_division=0
        )
    )


# ==========================================
# RUN
# ==========================================
main()

Torch device: cuda
Fusion model loaded.
Classes: ['angry' 'disgust' 'fear' 'happy' 'neutral' 'ps' 'sad']
WAV files found: 5600
Dataset size: 4800
Test samples: 720


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Extracting speech features...
Speech shape: (720, 128, 180)
Generating text embeddings...
Text shape: (720, 768)
Running prediction...
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step

Accuracy: 1.0

Classification Report:

              precision    recall  f1-score   support

       angry       1.00      1.00      1.00       120
     disgust       1.00      1.00      1.00       120
        fear       1.00      1.00      1.00       120
       happy       1.00      1.00      1.00       120
     neutral       1.00      1.00      1.00       120
          ps       0.00      0.00      0.00         0
         sad       1.00      1.00      1.00       120

    accuracy                           1.00       720
   macro avg       0.86      0.86      0.86       720
weighted avg       1.00      1.00      1.00       720



---
## 📊 Part 4 — Display All Results

In [18]:
import os, glob, json as _json, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")

import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import torch
import tensorflow as tf
import scipy.special as _sp

from IPython.display import display, Image as IPImage

from sklearn.preprocessing import (
    LabelEncoder,
    label_binarize
)

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score,
    roc_curve,
    auc
)

from transformers import (
    DistilBertTokenizerFast,
    DistilBertModel,
    DistilBertForSequenceClassification,
)


# =========================================================
# CONSTANTS
# =========================================================
SEED         = 42
SR           = 22050
MAX_PAD      = 128
N_MFCC       = 40

TESS_DIR     = os.environ.get(
    "TESS_DIR",
    "/content/tess"
)

SPEECH_DIR   = "/content/project/models/speech_pipeline"
TEXT_DIR     = "/content/project/models/text_pipeline"
FUSION_DIR   = "/content/project/models/fusion_pipeline"

RESULTS_AT   = "/content/project/Results/accuracy_tables"
RESULTS_PL   = "/content/project/Results/plots"

TORCH_DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

EMOTIONS = [
    "angry",
    "disgust",
    "fear",
    "happy",
    "neutral",
    "ps",
    "sad"
]

os.makedirs(RESULTS_AT, exist_ok=True)
os.makedirs(RESULTS_PL, exist_ok=True)

EMO_COLORS = {
    "angry": "#E63946",
    "disgust": "#6A4C93",
    "fear": "#F4A261",
    "happy": "#2EC4B6",
    "neutral": "#8D99AE",
    "ps": "#FFB703",
    "sad": "#457B9D",
}

MODEL_COLORS = [
    "#4C72B0",
    "#DD8452",
    "#59A14F"
]


# =========================================================
# ATTENTION LAYER
# =========================================================
class AttentionLayer(tf.keras.layers.Layer):

    def __init__(self, units=64, **kwargs):

        super().__init__(**kwargs)

        self.units = units

        self.W = tf.keras.layers.Dense(
            units,
            activation="tanh"
        )

        self.V = tf.keras.layers.Dense(1)

    def call(self, x):

        score = self.V(self.W(x))

        weight = tf.nn.softmax(
            score,
            axis=1
        )

        return tf.reduce_sum(
            weight * x,
            axis=1
        )

    def get_config(self):

        cfg = super().get_config()

        cfg.update({
            "units": self.units
        })

        return cfg


# =========================================================
# HELPERS
# =========================================================
def section(title):

    bar = "═" * 62

    print(f"\n{bar}\n  {title}\n{bar}")


def build_dataset():

    wav_files = glob.glob(
        os.path.join(TESS_DIR, "**", "*.wav"),
        recursive=True
    )

    records = []

    for fp in wav_files:

        folder = os.path.basename(
            os.path.dirname(fp)
        ).lower()

        emotion = folder.split("_")[-1]

        if emotion in EMOTIONS:

            records.append({
                "path": fp,
                "emotion": emotion
            })

    df = pd.DataFrame(records)

    _, test_df = train_test_split(
        df,
        test_size=0.15,
        random_state=SEED,
        stratify=df["emotion"]
    )

    return test_df.reset_index(drop=True)


def extract_speech_features(audio, sr=SR):

    mfcc = librosa.feature.mfcc(
        y=audio,
        sr=sr,
        n_mfcc=N_MFCC
    )

    delta = librosa.feature.delta(mfcc)

    mel = librosa.power_to_db(
        librosa.feature.melspectrogram(
            y=audio,
            sr=sr,
            n_mels=128,
            n_fft=2048,
            hop_length=512
        ),
        ref=np.max
    )

    chroma = librosa.feature.chroma_stft(
        y=audio,
        sr=sr,
        n_fft=2048,
        hop_length=512
    )

    sc = librosa.feature.spectral_contrast(
        y=audio,
        sr=sr,
        n_fft=2048,
        hop_length=512
    )

    zcr = librosa.feature.zero_crossing_rate(
        audio,
        hop_length=512
    )

    feat = np.vstack([
        mfcc,
        delta,
        mel,
        chroma,
        sc,
        zcr
    ]).T

    feat = np.pad(
        feat,
        (
            (0, max(0, MAX_PAD - feat.shape[0])),
            (0, 0)
        )
    )[:MAX_PAD, :180]

    feat = (
        (feat - feat.mean(0)) /
        (feat.std(0) + 1e-8)
    )

    return feat.astype(np.float32)


def save_df(df, filename):

    path = os.path.join(
        RESULTS_AT,
        filename
    )

    df.to_csv(path, index=False)

    display(df)

    print(f"  → saved: {path}")


def save_fig(fig, filename):

    path = os.path.join(
        RESULTS_PL,
        filename
    )

    fig.savefig(
        path,
        dpi=150,
        bbox_inches="tight"
    )

    plt.show()

    print(f"  → saved: {path}")


def plot_confusion_ax(ax, cm, classes, title):

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=classes,
        yticklabels=classes,
        linewidths=0.4,
        linecolor="white",
        ax=ax,
        cbar_kws={"shrink": 0.8}
    )

    ax.set_title(
        title,
        fontsize=10,
        fontweight="bold"
    )

    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")


# =========================================================
# TEST DATA
# =========================================================
section("0 · Building Test Split")

test_df = build_dataset()

print(f"  Test samples : {len(test_df)}")

le = LabelEncoder()

le.fit(EMOTIONS)

y_true = le.transform(
    test_df["emotion"].values
)

classes = le.classes_

results = {}


# =========================================================
# SPEECH MODEL
# =========================================================
section("1 · 🎙️ Speech Model")

sp_model_path = os.path.join(
    SPEECH_DIR,
    "speech_model.h5"
)

sp_lbl_path = os.path.join(
    SPEECH_DIR,
    "label_classes.npy"
)

if os.path.exists(sp_model_path):

    sp_model = tf.keras.models.load_model(
        sp_model_path,
        custom_objects={
            "AttentionLayer": AttentionLayer
        },
        compile=False
    )

    sp_le = LabelEncoder()

    sp_le.classes_ = np.load(
        sp_lbl_path,
        allow_pickle=True
    )

    print("  Extracting features...")

    X_sp = np.array([
        extract_speech_features(
            *librosa.load(
                r["path"],
                sr=SR,
                mono=True
            )
        )
        for _, r in test_df.iterrows()
    ])

    sp_score = sp_model.predict(
        X_sp,
        verbose=0
    )

    sp_pred = sp_le.inverse_transform(
        sp_score.argmax(1)
    )

    y_pred_sp = le.transform(sp_pred)

    sp_acc = accuracy_score(
        y_true,
        y_pred_sp
    )

    sp_f1 = f1_score(
        y_true,
        y_pred_sp,
        average="macro",
        zero_division=0
    )

    print(
        f"\n  Accuracy : {sp_acc*100:.2f}%   Macro-F1 : {sp_f1:.4f}"
    )

    sp_rep = classification_report(
        y_true,
        y_pred_sp,
        labels=np.arange(len(classes)),
        target_names=classes,
        zero_division=0,
        output_dict=True
    )

    sp_cm = confusion_matrix(
        y_true,
        y_pred_sp,
        labels=np.arange(len(classes))
    )

    results["Speech"] = {
        "acc": sp_acc,
        "f1": sp_f1,
        "report": sp_rep,
        "cm": sp_cm
    }

    save_df(
        pd.DataFrame(sp_rep).T,
        "speech_metrics.csv"
    )

else:

    print(f"  ✗ Not found: {sp_model_path}")


# =========================================================
# TEXT MODEL
# =========================================================
section("2 · 📝 Text Model")

tx_model_path = os.path.join(
    TEXT_DIR,
    "text_model"
)

tx_lbl_path = os.path.join(
    TEXT_DIR,
    "text_label_classes.npy"
)

if os.path.isdir(tx_model_path):

    tx_le = LabelEncoder()

    tx_le.classes_ = np.load(
        tx_lbl_path,
        allow_pickle=True
    )

    tx_tokenizer = DistilBertTokenizerFast.from_pretrained(
        tx_model_path
    )

    tx_model = DistilBertForSequenceClassification.from_pretrained(
        tx_model_path,
        num_labels=len(tx_le.classes_)
    ).to(TORCH_DEVICE)

    tx_model.eval()

    def path_to_word(fp):

        parts = os.path.splitext(
            os.path.basename(fp)
        )[0].split("_")

        return parts[-1]

    texts = [
        path_to_word(r["path"])
        for _, r in test_df.iterrows()
    ]

    tx_logits_list = []

    print("  Running inference...")

    for i in range(0, len(texts), 64):

        enc = tx_tokenizer(
            texts[i:i+64],
            max_length=32,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        ).to(TORCH_DEVICE)

        with torch.no_grad():

            tx_logits_list.append(
                tx_model(**enc).logits.cpu().numpy()
            )

    tx_logits = np.vstack(tx_logits_list)

    tx_score = _sp.softmax(
        tx_logits,
        axis=1
    )

    tx_pred = tx_le.inverse_transform(
        tx_score.argmax(1)
    )

    y_pred_tx = le.transform(tx_pred)

    tx_acc = accuracy_score(
        y_true,
        y_pred_tx
    )

    tx_f1 = f1_score(
        y_true,
        y_pred_tx,
        average="macro",
        zero_division=0
    )

    print(
        f"\n  Accuracy : {tx_acc*100:.2f}%   Macro-F1 : {tx_f1:.4f}"
    )

    tx_rep = classification_report(
        y_true,
        y_pred_tx,
        labels=np.arange(len(classes)),
        target_names=classes,
        zero_division=0,
        output_dict=True
    )

    tx_cm = confusion_matrix(
        y_true,
        y_pred_tx,
        labels=np.arange(len(classes))
    )

    results["Text"] = {
        "acc": tx_acc,
        "f1": tx_f1,
        "report": tx_rep,
        "cm": tx_cm
    }

    save_df(
        pd.DataFrame(tx_rep).T,
        "text_metrics.csv"
    )

else:

    print(f"  ✗ Not found: {tx_model_path}")


# =========================================================
# FUSION MODEL
# =========================================================
section("3 · 🔀 Fusion Model")

fu_model_path = os.path.join(
    FUSION_DIR,
    "fusion_model.h5"
)

fu_lbl_path = os.path.join(
    FUSION_DIR,
    "fusion_label_classes.npy"
)

if os.path.exists(fu_model_path):

    fu_model = tf.keras.models.load_model(
        fu_model_path,
        custom_objects={
            "AttentionLayer": AttentionLayer
        },
        compile=False
    )

    fu_le = LabelEncoder()

    fu_le.classes_ = np.load(
        fu_lbl_path,
        allow_pickle=True
    )

    if "X_sp" not in dir():

        X_sp = np.array([
            extract_speech_features(
                *librosa.load(
                    r["path"],
                    sr=SR,
                    mono=True
                )
            )
            for _, r in test_df.iterrows()
        ])

    print("  Generating DistilBERT embeddings...")

    bert_base = DistilBertModel.from_pretrained(
        "distilbert-base-uncased"
    ).to(TORCH_DEVICE)

    bert_base.eval()

    fu_tok = DistilBertTokenizerFast.from_pretrained(
        "distilbert-base-uncased"
    )

    X_tx_fu = []

    for i in range(0, len(test_df), 64):

        batch = ["word"] * min(
            64,
            len(test_df) - i
        )

        enc = fu_tok(
            batch,
            max_length=32,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        ).to(TORCH_DEVICE)

        with torch.no_grad():

            X_tx_fu.append(
                bert_base(**enc)
                .last_hidden_state[:, 0, :]
                .cpu()
                .numpy()
            )

    X_tx_fu = np.vstack(X_tx_fu)

    fu_score = fu_model.predict(
        [X_sp, X_tx_fu],
        verbose=0
    )

    fu_pred = fu_le.inverse_transform(
        fu_score.argmax(1)
    )

    y_pred_fu = le.transform(fu_pred)

    fu_acc = accuracy_score(
        y_true,
        y_pred_fu
    )

    fu_f1 = f1_score(
        y_true,
        y_pred_fu,
        average="macro",
        zero_division=0
    )

    print(
        f"\n  Accuracy : {fu_acc*100:.2f}%   Macro-F1 : {fu_f1:.4f}"
    )

    fu_rep = classification_report(
        y_true,
        y_pred_fu,
        labels=np.arange(len(classes)),
        target_names=classes,
        zero_division=0,
        output_dict=True
    )

    fu_cm = confusion_matrix(
        y_true,
        y_pred_fu,
        labels=np.arange(len(classes))
    )

    results["Fusion"] = {
        "acc": fu_acc,
        "f1": fu_f1,
        "report": fu_rep,
        "cm": fu_cm
    }

    save_df(
        pd.DataFrame(fu_rep).T,
        "fusion_metrics.csv"
    )

else:

    print(f"  ✗ Not found: {fu_model_path}")


# =========================================================
# COMPARISON TABLE
# =========================================================
section("4 · 📊 Model Comparison")

comp_df = pd.DataFrame([
    {
        "Model": name,
        "Accuracy": f"{r['acc']*100:.2f}%",
        "Macro F1": f"{r['f1']:.4f}"
    }
    for name, r in results.items()
])

save_df(
    comp_df,
    "model_comparison.csv"
)

print("\n" + "═"*62)
print("  ✅ All results generated successfully.")
print("═"*62)


══════════════════════════════════════════════════════════════
  0 · Building Test Split
══════════════════════════════════════════════════════════════
  Test samples : 720

══════════════════════════════════════════════════════════════
  1 · 🎙️ Speech Model
══════════════════════════════════════════════════════════════
  Extracting features...

  Accuracy : 91.81%   Macro-F1 : 0.9190


,precision,recall,f1-score,support
angry,0.991228,0.941667,0.965812,120.000000
disgust,0.731707,1.000000,0.845070,120.000000
fear,0.937500,1.000000,0.967742,120.000000
happy,0.942308,0.816667,0.875000,120.000000
neutral,1.000000,0.975000,0.987342,120.000000
ps,0.000000,0.000000,0.000000,0.000000
sad,1.000000,0.775000,0.873239,120.000000
accuracy,0.918056,0.918056,0.918056,0.918056
macro avg,0.800392,0.786905,0.787744,720.000000
weighted avg,0.933791,0.918056,0.919034,720.000000


  → saved: /content/project/Results/accuracy_tables/speech_metrics.csv

══════════════════════════════════════════════════════════════
  2 · 📝 Text Model
══════════════════════════════════════════════════════════════


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

  Running inference...

  Accuracy : 16.67%   Macro-F1 : 0.0476


...


  Accuracy : 95.97%   Macro-F1 : 0.9594


,precision,recall,f1-score,support
angry,0.991228,0.941667,0.965812,120.000000
disgust,0.882353,1.000000,0.937500,120.000000
fear,1.000000,0.975000,0.987342,120.000000
happy,0.983607,1.000000,0.991736,120.000000
neutral,1.000000,0.841667,0.914027,120.000000
ps,0.000000,0.000000,0.000000,0.000000
sad,0.923077,1.000000,0.960000,120.000000
accuracy,0.959722,0.959722,0.959722,0.959722
macro avg,0.825752,0.822619,0.822345,720.000000
weighted avg,0.963377,0.959722,0.959403,720.000000


  → saved: /content/project/Results/accuracy_tables/fusion_metrics.csv

══════════════════════════════════════════════════════════════
  4 · 📊 Model Comparison
══════════════════════════════════════════════════════════════


,Model,Accuracy,Macro F1
0,Speech,91.81%,0.9190
1,Text,16.67%,0.0476
2,Fusion,95.97%,0.9594


  → saved: /content/project/Results/accuracy_tables/model_comparison.csv

══════════════════════════════════════════════════════════════
  ✅ All results generated successfully.
══════════════════════════════════════════════════════════════
